# ChefBot AI - Dallo scatto alla ricetta

Progetto finale Modulo 5.

ChefBot è un assistente digitale che:
1. **riconosce il piatto** da una foto (Computer Vision, transfer learning su Food101);
2. mostra una **scheda informativa** con ingredienti e valori nutrizionali presi da un database JSON;
3. permette una **ricerca per umore**: l'utente scrive una frase e il sistema propone il piatto semanticamente più vicino (embeddings + similarità coseno).

> Consigliato eseguire su Google Colab con GPU (Runtime → Cambia tipo di runtime → GPU).

## 0. Setup

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.exists("Epicode-modulo5"):
    !git clone https://github.com/GioForna/Epicode-modulo5.git
    %cd Epicode-modulo5
    !pip install -q sentence-transformers gradio
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow_datasets as tfds
from sklearn.metrics import classification_report, confusion_matrix

from src import vision
from src.config import SELECTED_CLASSES, IMG_SIZE, MODELS_DIR, MODEL_PATH
from src.knowledge import load_knowledge_base, lookup, format_card
from src.semantic_search import MoodSearch, dish_to_text, cosine_similarity

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
MODELS_DIR.mkdir(exist_ok=True)

## 1. Dataset Food101

Carichiamo Food101 da TensorFlow Datasets (101 classi, 750 immagini di train e 250 di validation per classe).
Per il progetto usiamo un subset di **20 classi**, scelte in modo da avere piatti molto diversi tra loro (dolci, insalate, zuppe, primi, street food...) così da rendere interessante anche la ricerca semantica.

In [ ]:
(ds_train_raw, ds_test_raw), ds_info = vision.load_food101()
all_class_names = ds_info.features["label"].names

print("Classi totali:", ds_info.features["label"].num_classes)
print("Train:", ds_info.splits["train"].num_examples, "| Validation:", ds_info.splits["validation"].num_examples)
print("\nClassi selezionate:")
for c in SELECTED_CLASSES:
    print(f"  {all_class_names.index(c):3d} -> {c}")

In [ ]:
# Qualche esempio delle classi selezionate
selected_ids = tf.constant([all_class_names.index(c) for c in SELECTED_CLASSES], dtype=tf.int64)
examples = ds_train_raw.filter(lambda img, lbl: tf.reduce_any(lbl == selected_ids)).take(12)

plt.figure(figsize=(14, 9))
for i, (img, lbl) in enumerate(examples):
    plt.subplot(3, 4, i + 1)
    plt.imshow(img)
    plt.title(all_class_names[int(lbl)])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 2. Preprocessing

- rimappiamo le etichette originali (0-100) sugli indici del subset (0-19) con una `StaticHashTable` e filtriamo le altre classi;
- ridimensioniamo a 224x224;
- usiamo `cache()` così Food101 viene letto e filtrato una volta sola, poi `shuffle`, `batch` e `prefetch`.

La data augmentation (flip, rotazione, zoom, contrasto) è dentro il modello, quindi è attiva solo in training.

In [ ]:
table = vision.build_label_table(all_class_names, SELECTED_CLASSES)
train_ds = vision.prepare_dataset(ds_train_raw, table, training=True)
test_ds = vision.prepare_dataset(ds_test_raw, table, training=False)

for images, labels in train_ds.take(1):
    print(images.shape, images.dtype, labels[:10].numpy())

## 3. Modello: transfer learning con EfficientNetB0

- backbone **EfficientNetB0** pre-addestrato su ImageNet, con i pesi **congelati**;
- classificatore personalizzato: GlobalAveragePooling → BatchNorm → Dropout → Dense(256) → Dropout → Dense(20, softmax).

Nel modulo `src/vision.py` è disponibile anche `MobileNetV2` (`backbone="mobilenet"`).

In [ ]:
NUM_CLASSES = len(SELECTED_CLASSES)
model, base_model = vision.build_model(NUM_CLASSES, backbone="efficientnet")
vision.compile_model(model, learning_rate=1e-3)
model.summary()

### 3.1 Fase 1 - training del solo classificatore

In [ ]:
EPOCHS = 10
history = model.fit(train_ds, validation_data=test_ds, epochs=EPOCHS, callbacks=vision.get_callbacks())

### 3.2 Fase 2 - fine-tuning

Una volta che la testa è addestrata, sblocchiamo gli ultimi 30 layer di EfficientNet (BatchNorm escluse) e continuiamo con un learning rate molto basso, per adattare le feature di alto livello al dominio del cibo senza "rompere" quelle già apprese.

In [ ]:
FINE_TUNE_EPOCHS = 5
vision.unfreeze_top_layers(base_model, n_layers=30)
vision.compile_model(model, learning_rate=1e-5)

start = len(history.history["loss"])
history_ft = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=start + FINE_TUNE_EPOCHS,
    initial_epoch=start,
    callbacks=vision.get_callbacks(),
)

## 4. Valutazione

In [ ]:
hist = {k: history.history[k] + history_ft.history.get(k, []) for k in history.history}

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, metric in zip(axes, ["accuracy", "loss"]):
    ax.plot(hist[metric], label="train")
    ax.plot(hist[f"val_{metric}"], label="validation")
    ax.axvline(start - 0.5, color="gray", linestyle="--", label="inizio fine-tuning")
    ax.set_title(metric)
    ax.set_xlabel("epoca")
    ax.legend()
plt.show()

loss, acc, top3 = model.evaluate(test_ds, verbose=0)
print(f"Accuracy: {acc:.3f} | Top-3 accuracy: {top3:.3f}")

In [ ]:
y_true, y_pred = [], []
for images, labels in test_ds:
    y_true.append(labels.numpy())
    y_pred.append(np.argmax(model.predict_on_batch(images), axis=1))
y_true, y_pred = np.concatenate(y_true), np.concatenate(y_pred)

print(classification_report(y_true, y_pred, target_names=SELECTED_CLASSES, digits=3))

In [ ]:
cm = confusion_matrix(y_true, y_pred, normalize="true")
plt.figure(figsize=(13, 11))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=SELECTED_CLASSES, yticklabels=SELECTED_CLASSES)
plt.xlabel("Predetto")
plt.ylabel("Reale")
plt.title("Matrice di confusione (normalizzata)")
plt.show()

# Coppie di classi che il modello confonde di più
off_diag = cm.copy()
np.fill_diagonal(off_diag, 0)
for idx in np.argsort(off_diag.ravel())[::-1][:5]:
    i, j = np.unravel_index(idx, cm.shape)
    print(f"{SELECTED_CLASSES[i]:>20} -> {SELECTED_CLASSES[j]:<20} {off_diag[i, j]:.1%}")

### Salvataggio

In [ ]:
model.save(MODEL_PATH)
vision.save_class_names()
print("Modello salvato in", MODEL_PATH)

## 5. Knowledge Retrieval

Food101 contiene solo immagini ed etichette, quindi abbiamo costruito un "dizionario della conoscenza" in `data/knowledge_base.json`: per ogni etichetta ci sono nome, origine, descrizione, ingredienti principali, calorie stimate per porzione, macronutrienti e alcuni tag.

La funzione `lookup()` riceve la predizione del modello e restituisce i metadati corrispondenti.

In [ ]:
kb = load_knowledge_base()
print("Piatti nel database:", len(kb))
assert set(kb) == set(SELECTED_CLASSES), "Ogni classe deve avere la sua scheda"

print(json.dumps(lookup("baklava"), ensure_ascii=False, indent=2))

### Pipeline completa: foto → predizione → scheda

In [ ]:
from IPython.display import Markdown, display

class_names = vision.load_class_names()

for images, labels in test_ds.unbatch().shuffle(500).batch(3).take(1):
    for img, lbl in zip(images, labels):
        preds = vision.predict(model, img.numpy(), class_names, top_k=3)
        label, conf = preds[0]

        plt.figure(figsize=(4, 4))
        plt.imshow(img.numpy())
        plt.title(f"Reale: {class_names[int(lbl)]}\nPredetto: {label} ({conf:.0%})")
        plt.axis("off")
        plt.show()

        display(Markdown(format_card(lookup(label), conf)))

## 6. Ricerca semantica ("Ricerca per Umore")

Per ogni piatto costruiamo un testo descrittivo (nome + descrizione + ingredienti + tag) e lo trasformiamo in un vettore con **Sentence-Transformers**.
Usiamo il modello multilingua `paraphrase-multilingual-MiniLM-L12-v2` perché le frasi degli utenti sono in italiano.

Quando l'utente scrive una frase, calcoliamo il suo embedding e la **similarità coseno** con tutti i piatti:

$$\cos(\mathbf{q}, \mathbf{d}) = \frac{\mathbf{q} \cdot \mathbf{d}}{\lVert\mathbf{q}\rVert \, \lVert\mathbf{d}\rVert}$$

Il piatto con il punteggio più alto è quello proposto.

In [ ]:
search = MoodSearch(kb)
print("Embeddings:", search.embeddings.shape)
print("\nEsempio di testo indicizzato:\n", dish_to_text(kb["churros"]))

In [ ]:
queries = [
    "voglio qualcosa di fresco e leggero",
    "dolce con miele",
    "ho freddo, mi serve qualcosa di caldo e confortante",
    "ho voglia di pesce",
    "un piatto piccante ed esotico",
    "qualcosa di goloso da mangiare con le mani",
    "colazione dolce",
]

for q in queries:
    results = search.search(q, top_k=3)
    print(f"\n> {q}")
    for r in results:
        print(f"   {r['nome']:<28} {r['score']:.3f}")

### Confronto con una ricerca per parole esatte

Una ricerca "classica" cerca la parola nel nome del piatto: con "dolce con miele" non trova nulla, perché nessun nome contiene quelle parole. Gli embeddings invece catturano il significato.

In [ ]:
def keyword_search(query):
    words = query.lower().split()
    return [v["nome"] for v in kb.values() if any(w in v["nome"].lower() for w in words if len(w) > 3)]

for q in ["dolce con miele", "voglio qualcosa di fresco e leggero"]:
    print(f"{q!r}")
    print("   parole esatte:", keyword_search(q) or "nessun risultato")
    print("   semantica    :", [r["nome"] for r in search.search(q)])

In [ ]:
# Similarità tra i piatti nello spazio degli embeddings
emb = search.embeddings / np.linalg.norm(search.embeddings, axis=1, keepdims=True)
sim = emb @ emb.T
names = [kb[k]["nome"] for k in search.labels]

plt.figure(figsize=(13, 11))
sns.heatmap(sim, xticklabels=names, yticklabels=names, cmap="viridis")
plt.title("Similarità coseno tra i piatti")
plt.show()

## 7. Demo interattiva

L'app Gradio (`app.py`) mette insieme le tre parti: caricamento foto con scheda del piatto e ricerca per umore.

In [ ]:
from app import demo
demo.launch(share=IN_COLAB)

## 8. Conclusioni

- Il transfer learning con EfficientNetB0 congelato permette di ottenere buoni risultati su 20 classi di Food101 in pochi minuti di training; il fine-tuning degli ultimi layer aggiunge qualche punto di accuracy.
- Le confusioni principali sono tra piatti visivamente simili (es. insalate tra loro, ramen e miso soup).
- Il knowledge base JSON collega la predizione del modello a informazioni utili per l'utente.
- La ricerca semantica con Sentence-Transformers trova piatti coerenti con la richiesta anche quando le parole non compaiono nel nome.

**Possibili sviluppi:** estendere a tutte le 101 classi, valori nutrizionali da un database ufficiale (es. CREA / USDA), soglia di confidenza per rispondere "non lo so", combinare foto e testo nella stessa ricerca.